# 02 -- Embedding Inspection

Create synthetic embeddings for each dialect, inspect nearest neighbours across
varieties, and visualise the embedding spaces using PCA and distance matrices.

**Key modules:** `embeddings.base`, `embeddings.alignment`, `types`, `constants`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import cosine, cdist
from eigendialectos.constants import DialectCode, DIALECT_NAMES
from eigendialectos.types import EmbeddingMatrix

## Create Synthetic Embeddings

Generate random embedding matrices for demonstration. In practice, these would
come from `EmbeddingModel.train()` on real corpus data.

In [ ]:
rng = np.random.default_rng(42)
dim, vocab_size = 50, 200
vocab = [f"word_{i}" for i in range(vocab_size)]

# Base embedding (Peninsular)
base = rng.standard_normal((dim, vocab_size)).astype(np.float64)

embeddings = {}
for code in DialectCode:
    if code == DialectCode.ES_PEN:
        data = base.copy()
    else:
        noise_scale = 0.1 + 0.15 * rng.random()
        data = base + rng.standard_normal((dim, vocab_size)) * noise_scale
    embeddings[code] = EmbeddingMatrix(data=data, vocab=vocab, dialect_code=code)

print(f"Embedding spaces: {len(embeddings)}")
print(f"Dimensionality: {dim}, Vocabulary: {vocab_size}")

## Nearest Neighbours Across Varieties

For a given word, check how far its embedding drifts across dialects.

In [ ]:
ref = embeddings[DialectCode.ES_PEN]
query_words = ["word_0", "word_10", "word_50", "word_100"]

for word in query_words:
    idx = vocab.index(word)
    ref_vec = ref.data[:, idx]
    print(f"\n--- {word} ---")
    distances = []
    for code in sorted(embeddings.keys(), key=lambda c: c.value):
        emb = embeddings[code]
        vec = emb.data[:, idx]
        d = float(np.linalg.norm(ref_vec - vec))
        cos_d = float(cosine(ref_vec, vec)) if np.linalg.norm(vec) > 0 else 0.0
        distances.append((code.value, d, cos_d))
    for name, d, cos_d in sorted(distances, key=lambda x: x[1]):
        print(f"  {name}: L2={d:.4f}  cosine_dist={cos_d:.4f}")

## Inter-Dialect Distance Matrix

Average L2 distance between corresponding word vectors across all dialect pairs.

In [ ]:
codes = sorted(embeddings.keys(), key=lambda c: c.value)
n = len(codes)
dist_matrix = np.zeros((n, n))

for i, ci in enumerate(codes):
    for j, cj in enumerate(codes):
        if i < j:
            diff = embeddings[ci].data - embeddings[cj].data
            avg_dist = float(np.mean(np.linalg.norm(diff, axis=0)))
            dist_matrix[i, j] = avg_dist
            dist_matrix[j, i] = avg_dist

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(dist_matrix, cmap="YlOrRd")
ax.set_xticks(range(n))
ax.set_xticklabels([c.value for c in codes], rotation=45, ha="right")
ax.set_yticks(range(n))
ax.set_yticklabels([c.value for c in codes])
fig.colorbar(im, ax=ax, label="Mean L2 Distance")
ax.set_title("Pairwise Embedding Distance (Pre-Alignment)")
plt.tight_layout()
plt.show()

## PCA Visualisation

Project all embeddings into 2D via PCA for visual comparison.

In [ ]:
from sklearn.decomposition import PCA

# Stack a sample of word vectors from each dialect
n_sample = 50
all_vecs = []
all_labels = []
for code in codes:
    vecs = embeddings[code].data[:, :n_sample].T  # (n_sample, dim)
    all_vecs.append(vecs)
    all_labels.extend([code.value] * n_sample)

all_vecs = np.vstack(all_vecs)
pca = PCA(n_components=2)
projected = pca.fit_transform(all_vecs)

fig, ax = plt.subplots(figsize=(10, 8))
for code in codes:
    mask = [l == code.value for l in all_labels]
    pts = projected[mask]
    ax.scatter(pts[:, 0], pts[:, 1], s=15, alpha=0.6, label=code.value)

ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
ax.set_title("PCA of Word Embeddings by Dialect")
ax.legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.show()